In [ ]:
import pandas as pd
import pathlib

def generate_events(energies, nevent=1000000, level="hard"):
    
    def generate_events_for_energy(energy, nevent, level):
        # Setup Pythia
        pythia = pythia8.Pythia()

        # Set random seeds
        pythia.readString("Random:setSeed=on")
        pythia.readString("Random:seed=0")

        # Define beams (frameType = 2 for fixed target collisions)
        pythia.readString("Beams:frameType = 2")
        pythia.readString(f"Beams:idA = 14")
        pythia.readString(f"Beams:eA = {energy}")
        pythia.readString("Beams:eB = 0")

        # Use the nuclear PDF nCTEQ15 for Tungsten (A=184, Z=74)
        pdffile = "nCTEQ15FullNuc_56_26_0000.dat"
        pythia.readString(f"PDF:pSet=lhagrid1:{pdffile}")

        # Activate necessary options for Pythia
        pythia.readString("PDF:lepton = on")
        pythia.readString("TimeShower:QEDshowerByL = on")

        # Fix Renormalization/Factorization scale
        pythia.readString("SigmaProcess:factorScale2=6")
        pythia.readString("SigmaProcess:renormScale2=6")

        # Minimal phase space cuts
        pythia.readString("PhaseSpace:mHatMin=1")
        pythia.readString("PhaseSpace:pTHatMin=0")
        pythia.readString("PhaseSpace:pTHatMinDiverge=0")

        # Define process (neutrino charge current collision)
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")

        # Decide if shower and hadronization should be included
        if level == "hard":
            pythia.readString("HadronLevel:all=off")
            pythia.readString("PartonLevel:all=off")
        elif level == "parton":
            pythia.readString("HadronLevel:all=off")
            pythia.readString("PartonLevel:all=on")
        elif level == "hadron":
            pythia.readString("HadronLevel:all=on")
            pythia.readString("PartonLevel:all=on")
        else:
            print(f"Error: level = {level} is not a valid option")
            return pd.DataFrame()

        # Initialize
        pythia.init()

        # List to store events data for DataFrame
        event_data = []

        # Loop over events
        for ievent in range(nevent):
            # Generate next event
            if not pythia.next():
                continue

            particles = pythia.process if level == "hard" else pythia.event

            # Loop through particles in event
            for iparticle, particle in enumerate(particles):
                # Reject non-final state particles
                if particle.status() < 0:
                    continue
                # Collect particle data
                pid = particle.id()
                px, py, pz, e = particle.px(), particle.py(), particle.pz(), particle.e()
                parent_pid = particle.mother1()  

                # Append particle data as a new row
                event_data.append([ievent, iparticle, energy, pid, px, py, pz, e, parent_pid])

        # Convert collected event data to DataFrame
        columns = ["ievent", "iparticle", "truth_energy", "pid", "px", "py", "pz", "e", "parent_pid"]
        return pd.DataFrame(event_data, columns=columns)

    ##Output directory
    output_dir = pathlib.Path("output_events")
    output_dir.mkdir(exist_ok=True)

    # Loop over each energy and generate events, saving each DataFrame as a CSV file
    for energy in energies:
        df = generate_events_for_energy(energy, nevent, level)
        csv_file = output_dir / f"events_{energy}.csv"
        df.to_csv(csv_file, index=False)
        print(f"Saved events for energy {energy} to {csv_file}")


energies = [
    14.219093, 17.900778, 22.535744, 28.370820, 35.716747, 44.964720, 56.607229,
    71.264279, 89.716412, 112.946271, 142.190930, 179.007775, 225.357437,
    283.708205, 357.167468, 449.647202, 566.072289, 712.642790, 897.164117,
    1129.462706, 1421.909302, 1790.077754, 2253.574373, 2837.082046, 3571.674683,
    4496.472021, 5660.722891, 7126.427896, 8971.641174
] # List of energies
generate_events(energies, nevent=10**7, level="hadron")
